# W11A · 8A · attention을 직접 계산하기

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-10**

예상 → 계산 → 실행 → 한 조건 수정 → 이유 설명 순서로 기록하세요. 자신의 Colab 사본을 저장하고 첫 셀부터 실행합니다. 기본값은 CPU이며 공식 데이터 자동 준비·캐시 재사용을 지원합니다. 설치 갱신 안내가 나오면 런타임 재시작 뒤 처음부터 실행하세요.

`None`은 자신의 답을 적는 자리이며 예제 실행 변수와 분리돼 있습니다. 예제가 끝까지 실행됐다고 자신의 문제 풀이가 완료된 것은 아닙니다.


In [ ]:
from __future__ import annotations
import os, sys, subprocess
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w11a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps,neural] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'], check=True)
    for package, wanted in [('tensorflow','2.21.0'), ('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package], '__version__', None) != wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from IPython.display import display
from luna_recsys.autoencoder_recommenders import (
    toy_ratings, fixed_ae_forward, masked_metrics, loss_comparison,
    build_fixed_autoencoder, prepare_autoencoder_data, oriented_arrays,
    fit_autoencoder, predict_autoencoder, evaluate_autoencoder, recommend_autoencoder,
)
from luna_recsys.autoencoder_explainer import (
    ae_view, export_ae_csv, build_autoencoder_explainer,
)
from luna_recsys.datasets import prepare_movielens
print('Python',sys.version.split()[0], 'NumPy',np.__version__, 'TensorFlow',tf.__version__, 'Keras',keras.__version__)
print('수업 ref:',COURSE_VERSION,'CPU 실행; 학습 자료 모드를 결과와 함께 확인합니다.')

from typing import Any, Callable


In [ ]:
from luna_recsys.sequential_recommenders import (
    scaled_attention, pad_history, prefix_pairs, prepare_sequence_data,
    build_sequence_model, sequence_train_step, fit_sequence,
    select_candidates, rollout, ranking_metrics, evaluate_sequence,
)
from luna_recsys.transformer_explainer import (
    attention_view, export_attention, sequence_view, export_sequence, build_transformer_explainer,
)


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/07-qkv-scores.png" alt="Q 한 행과 K의 내적" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/07-qkv-scores.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


### 本문 실행 정의 · `scaled_attention`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: ca60dfeab39c6cbf5ca1f5ab9650fedd4d9045be811f6088f1b864b71a1ddc27
def scaled_attention(q: Any, k: Any, v: Any, allowed: Any = None) -> dict:
    """(T,D),(S,D),(S,V) -> scores/weights/output; True permits a key.

    A fully masked query has zero weights/output. Does not learn or mutate inputs.
    Example: scaled_attention(np.eye(2),np.eye(2),np.eye(2))['weights'].
    """
    q, k, v = [np.asarray(x, dtype=float) for x in (q, k, v)]
    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2 or q.shape[1] != k.shape[1] or len(k) != len(v) or not q.shape[1]:
        raise ValueError('Q/K/V must have compatible two-dimensional shapes.')
    if not all(np.isfinite(x).all() for x in (q, k, v)):
        raise ValueError('Q/K/V must be finite.')
    scores = q @ k.T / np.sqrt(q.shape[1])
    mask = np.ones(scores.shape, bool) if allowed is None else np.broadcast_to(np.asarray(allowed, bool), scores.shape)
    masked = np.where(mask, scores, -1e9)
    unnormalized = np.exp(masked - masked.max(axis=-1, keepdims=True)) * mask
    denominator = unnormalized.sum(axis=-1, keepdims=True)
    weights = np.divide(unnormalized, denominator, out=np.zeros_like(unnormalized), where=denominator > 0)
    return {'scores': scores, 'weights': weights, 'output': weights @ v}


## 주어진 수치를 한 단계씩 확인하기

입력 Q/K는 `(3,2)`, V도 `(3,2)`입니다. 반환 사전의 scores/weights는 `(3,3)`, output은 `(3,2)`입니다. 이 셀은 학습하지 않습니다.


In [ ]:
Q=K=np.array([[1.,0.],[0.,1.],[1.,1.]])
V=np.array([[2.,0.],[0.,2.],[2.,2.]])
trace_attention=scaled_attention(Q,K,V)
for name,value in trace_attention.items():
    print(name,value.shape);display(pd.DataFrame(value))


## T01 · 직접 설명하고 수정하기

두 번째 query `[0,1]`의 내적·스케일 점수·softmax·출력을 먼저 적고 코드로 확인하세요.


In [ ]:
my_t01 = None


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/09-two-masks.png" alt="패딩·인과 마스크" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/09-two-masks.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


In [ ]:
causal_mask=np.tril(np.ones((3,3),bool))
causal_trace=scaled_attention(Q,K,V,allowed=causal_mask)
print('인과 마스크 후 첫 행:',causal_trace['weights'][0],causal_trace['output'][0])
padding_trace=scaled_attention(Q,K,V,allowed=np.tile([True,True,False],(3,1)))
print('패딩 제외:',padding_trace['weights'])


## T02 · 직접 설명하고 수정하기

query A가 뒤의 B/C를 읽지 못하도록 했을 때 첫 출력은 무엇인가요? 패딩 마스크와 같은 뜻인지 설명하세요.


In [ ]:
my_t02 = None


## 같은 투영을 Keras에 넣어 검산하기

아래 한 헤드의 Q/K/V와 출력 투영을 단위행렬로 고정합니다. 학습된 attention 실험과 구별하세요.


In [ ]:
mha=keras.layers.MultiHeadAttention(num_heads=1,key_dim=2,use_bias=False)
_ = mha(Q[None],V[None],key=K[None])
eye=np.eye(2,dtype='float32')
mha.set_weights([eye.reshape(2,1,2)]*3+[eye.reshape(1,2,2)])
keras_output,keras_attention=mha(Q[None],V[None],key=K[None],return_attention_scores=True)
print('고정 Keras 출력:',keras_output.numpy()[0])
np.testing.assert_allclose(keras_output.numpy()[0],trace_attention['output'],atol=1e-6)


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/10-multihead-shapes.png" alt="헤드당 폭과 최종 투영" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/10-multihead-shapes.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


## T03 · 직접 설명하고 수정하기

입력폭32·헤드2·key_dim32의 내부 Q 크기와 최종 출력폭을 설명하세요. key_dim을16으로 바꾸면 같은 모형인가요?


In [ ]:
my_t03 = None


## 함수 수정 → 앱 계산표 → CSV

본문 `scaled_attention`을 읽고 가중합 부분을 찾습니다. 아래 앱은 이 함수를 명시적으로 전달받습니다. 수정 후 함수를 재실행하고 앱도 다시 조립하세요.


In [ ]:
app_preview=attention_view(0,False,attention_fn=scaled_attention)
display(app_preview[1]);display(app_preview[2])
csv_path=export_attention(0,False,attention_fn=scaled_attention)
assert np.allclose(pd.read_csv(csv_path).weight,app_preview[1].weight)
demo=build_transformer_explainer(attention_fn=scaled_attention)
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
else:
    print('실제 브라우저 조작은 별도 검증합니다.')


## T04 · 직접 설명하고 수정하기

앱에서 query와 인과 마스크를 바꿔 표·그림·CSV를 확인하세요. 본문 함수의 변경이 앱에 전달되는 인수를 찾고 두 출력 성분의 차이를 설명하세요.


In [ ]:
my_t04 = None


In [ ]:
np.testing.assert_allclose(trace_attention['output'][0],[1.6044483708,1.1977758146])
np.testing.assert_allclose(trace_attention['weights'].sum(1),1)
assert not causal_trace['weights'][np.triu_indices(3,1)].any()
print('W11A_8A_RUNTIME_PASS: default demo/API checks; learner answers separate')


## 점검 퀴즈

1. 7장의 평점 복원과 8장의 다음 아이템 예측은 입력·정답·손실이 어떻게 다른가요?
2. 중심화된 관측값이 0일 때도 손실에 포함해야 하는 이유는 무엇인가요?
3. Q가 3×2, K가 3×2, V가 3×2이면 attention 가중치와 출력의 크기는 각각 무엇인가요?
4. attention 가중치의 한 행 합이 1인 이유와 V에 곱하는 이유는 무엇인가요?
5. 패딩 마스크와 인과 마스크는 무엇을 다르게 가리나요?
6. 정답 아이템 확률이 .5일 때 교차엔트로피는 얼마이며, 학습은 무엇을 바꾸나요?
7. 반복 추천에서 원본 ID와 내부 ID, 실제 다음 정답은 각각 어디에 사용하나요?
8. 추천 7개 중 정답 10개와 겹치는 아이템이 2개일 때 P/R/F1을 구하고 평가 의미를 설명하세요.


## 기억할 것과 참고자료

입력·정답·중간값·실제 갱신·평가를 연결하세요.

- 주교재 임일(2025), 8장 전체 pp.158–176, 연습8-1. 독자 수치와 보완 구현을 구분합니다.
- [Attention 원 논문](https://arxiv.org/html/1706.03762v7)
- [Keras MultiHeadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/)
- [MovieLens 이용 조건](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt)
